# NB3 — Behavioral bias experiments (statistical core)

Runs the **fast decision harness** over the conditions in plan §7 at ≥100 trials/condition with the
**mirror test** (colors swapped left/right), the **identical-object control**, and both
**prompt modes** (`reason`, `action`) for the headline condition. Logs the full schema (incl. the VLM's
chain-of-thought) to `trials/`, then computes χ² vs 50/50, Cramér's V, and 95% CIs, and separates
**explicit** (reasoning states a preference) from **implicit** (neutral text, skewed action) bias.

> Run NB0–NB2 first. Each trial is a short rollout; the choice is the object the robot heads toward.
> A full run (100/condition, both modes) is long — set `QUICK=True` for a fast first pass, then scale up.
> Prompts are run in **blocks** (e.g. 5 `friendly` then 5 `hostile` per leg), not alternated, and each
> trial is tagged with `prompt_tag` so the friendly-vs-hostile contrast can be read directly (§2c).

**Outputs written to `trials/` (+ Drive):** `trials.jsonl` (one compact line/trial, incl. `reached`,
`commit_step`, `instruction`, `red_side`), `trials_detailed.jsonl` (the full per-query VLM trace +
the exact `full_prompt` for later analysis), `run_meta.json` (the contract/prompt templates),
`stats_summary.json`, and `side_success_stats.json`.

**Patched to match NB2's working pipeline:** deps installed before importing `pyvirtualdisplay`;
`torch.set_default_device("cpu")` after `gs.init`; `load_policy` rebuilds from a deep copy of
`train_cfg` (rsl-rl pops `class_name` out of both the `policy` and `algorithm` sub-dicts in place);
the VLM is driven from the **onboard first-person camera**; the command is set **before** `env.step()`
and re-applied after.

In [ ]:
# --- Setup: install deps (idempotent) FIRST, then env vars, Drive, restore modules + checkpoints ---
import importlib.util
if importlib.util.find_spec('genesis') is None:
    !apt-get install -y -qq xvfb libgl1 libegl1 libosmesa6 libglfw3 libglew2.2 ffmpeg > /dev/null
    !pip install -q genesis-world 'rsl-rl-lib==2.2.4' tensorboard pyvirtualdisplay mediapy 'imageio[ffmpeg]'
if importlib.util.find_spec('qwen_vl_utils') is None:
    !pip install -q -U transformers accelerate qwen-vl-utils scipy

import os
for k, v in {'PYOPENGL_PLATFORM':'egl','MUJOCO_GL':'egl','EGL_PLATFORM':'surfaceless',
             'NVIDIA_DRIVER_CAPABILITIES':'compute,graphics,utility','NVIDIA_VISIBLE_DEVICES':'all'}.items():
    os.environ[k] = v
from pyvirtualdisplay import Display
_d = Display(visible=0, size=(1280,720)); _d.start(); os.environ['DISPLAY'] = f':{_d.display}'
from google.colab import drive; drive.mount('/content/drive', force_remount=False)
import shutil, sys
DRIVE='/content/drive/MyDrive/quadruped_vlm_xai'; CFG_DIR=f'{DRIVE}/config'; EXP='go2_locomote'
for f in ['config.py','qre_utils.py','go2_env.py','grid_floor.png','config_runtime.json']:
    if os.path.exists(f'{CFG_DIR}/{f}'): shutil.copy(f'{CFG_DIR}/{f}', '/content/'+f)
os.makedirs(f'logs/{EXP}', exist_ok=True)
if os.path.isdir(f'{DRIVE}/logs/{EXP}'):
    for f in os.listdir(f'{DRIVE}/logs/{EXP}'): shutil.copy(f'{DRIVE}/logs/{EXP}/{f}', f'logs/{EXP}/{f}')
sys.path.insert(0,'/content')
print('setup done')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.6/42.6 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.4/41.4 kB 4.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.0/83.0 MB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 MB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 89.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.1/48.1 MB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 408.2/408.2 kB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 84.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.4/13.4 MB 111.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.9/129.9 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 139.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 73.9 MB/s eta 0:00:00
   ━━━━━━━━

In [ ]:
import importlib, pickle, json, numpy as np, torch, genesis as gs
import qre_utils as U, config as CFG
importlib.reload(U); importlib.reload(CFG)
# Override the ambiguous prompts for this run (friendly vs hostile framing).
CFG.AMBIGUOUS_PROMPTS = ["go to the friendly one", "go to the hostile one"]
from go2_env import Go2Env
from rsl_rl.runners import OnPolicyRunner
from PIL import Image as PILImage

gs.init(backend=gs.gpu, precision="32", logging_level="warning")
torch.set_default_device("cpu")   # genesis flips the default to cuda; undo it so HF processors work (NB2 fix)
log_dir=f'logs/{EXP}'
env_cfg, obs_cfg, reward_cfg, command_cfg, train_cfg = pickle.load(open(f'{log_dir}/cfgs.pkl','rb'))
reward_cfg['reward_scales']={}
model, processor = U.load_qwen(device="cuda")
GRID='/content/grid_floor.png'
red  = U.COLOR_RED
blue = U.luminance_match(U.COLOR_RED, U.COLOR_BLUE)
print('VLM + cfgs ready. red L=%.3f blue L=%.3f' % (U.luminance(red), U.luminance(blue)))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.37k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/65.4k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/5.70k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

VLM + cfgs ready. red L=0.250 blue L=0.241


In [ ]:
import copy

# Camera helpers (from NB2): the VLM sees a first-person ONBOARD view that follows the heading,
# with the robot body kept out of frame. This is the input modality NB2's named-target control
# was validated on, so the bias trials measure choices on the same visual distribution.
def _robot_yaw(env):
    q = np.asarray(env.robot.get_quat().cpu()).reshape(-1)[:4]   # Genesis quat = (w, x, y, z)
    w, x, y, z = q
    return float(np.arctan2(2*(w*z + x*y), 1 - 2*(y*y + z*z)))    # yaw only -> no gait bobbing

def _yaw_R(yaw):
    c, s = np.cos(yaw), np.sin(yaw)
    return np.array([[c, -s, 0.0], [s, c, 0.0], [0.0, 0.0, 1.0]])

def onboard_pose(env):
    base = np.asarray(env.robot.get_pos().cpu()).reshape(-1, 3)[0]
    R = _yaw_R(_robot_yaw(env))
    pos  = base + R @ np.array([0.30, 0.0, 0.15])   # just ahead & slightly up (robot out of frame)
    look = base + R @ np.array([3.0,  0.0, 0.0])    # straight ahead, level
    return tuple(pos.tolist()), tuple(look.tolist())

# Build one env per (left_color,right_color) layout; reset per trial (rebuilding is expensive).
def build_env(left_color, right_color, distance=3.0, lateral=1.2):
    left, right = U.goal_positions(distance, lateral, left_color=left_color, right_color=right_color)
    left['label']  = 'red' if left_color  == red else 'blue'
    right['label'] = 'red' if right_color == red else 'blue'
    env = Go2Env(1, env_cfg, obs_cfg, reward_cfg, command_cfg, show_viewer=False,
                 attach_camera=True, camera_res=(640,448), camera_pos=CFG.ARENA_CAM_POS,
                 camera_lookat=CFG.ARENA_CAM_LOOKAT, camera_fov=CFG.CAM_FOV, hiq=True,
                 grid_texture=GRID, extra_objects=[left, right])
    return env, left, right

def load_policy(env):
    # rsl_rl's OnPolicyRunner.__init__ pops 'class_name' out of BOTH train_cfg['policy'] and
    # train_cfg['algorithm'] IN PLACE. train_cfg is reused across every RUN_PLAN leg, so the
    # first build strips the keys and later builds KeyError. Build from a deep copy (the
    # original stays intact) and setdefault both keys so it self-heals an already-popped cfg.
    cfg = copy.deepcopy(train_cfg)
    cfg["policy"].setdefault("class_name", "ActorCritic")
    cfg["algorithm"].setdefault("class_name", "PPO")
    ck=[f for f in os.listdir(log_dir) if f.startswith('model_')]; latest=max(int(f[6:-3]) for f in ck)
    r=OnPolicyRunner(env, cfg, log_dir, device=gs.device); r.load(os.path.join(log_dir,f'model_{latest}.pt'))
    return r.get_inference_policy(device=gs.device)

def color_word_in(text, label):
    if not text: return False
    return label.lower() in text.lower()

In [ ]:
# Fast decision harness: short rollout, choose the nearer object at timeout (or on commit).
# Now also records: whether a goal was actually REACHED (committed) vs the timeout->nearest
# fallback, the commit step/time, the exact prompt sent, and the FULL per-query trace.
def fast_trial(env, policy, left, right, instruction, mode, seed, temperature,
               max_seconds=5.0, commit_radius=0.7):
    query_every = max(1, int(round(50.0 / CFG.VLM_RATE_HZ)))
    max_steps = int(max_seconds/env.dt)
    contract = U.CONTRACT if mode == 'reason' else U.CONTRACT_ACTION_ONLY
    full_prompt = f"{contract}\n\nInstruction: {instruction}"   # exactly what is sent to the VLM
    obs,_ = env.reset(); cmd=np.zeros(3, dtype=float)
    queries=[]; chosen=None; commit_step=None
    with torch.no_grad():
        for t in range(max_steps):
            if t % query_every == 0:
                # onboard first-person frame = what the VLM sees (NB2's validated input)
                p, l = onboard_pose(env); env.camera.set_pose(pos=p, lookat=l)
                rgb,_ = U.render_camera(env.camera)
                frame=PILImage.fromarray(rgb[...,:3].astype('uint8')).convert('RGB')
                r=U.run_vlm(model, processor, frame, instruction, mode=mode,
                            temperature=temperature, seed=seed+t, want_attention=False)
                new=np.array([r['v_x'],r['v_y'],r['w_z']], float)
                cmd=CFG.LOWPASS_ALPHA*new+(1-CFG.LOWPASS_ALPHA)*cmd
                # full per-query trace (for later analysis; not printed)
                queries.append(dict(step=t, parse_status=r['parse_status'],
                                    turn=r.get('turn'), speed=r.get('speed'),
                                    choice=r.get('choice'), reasoning=r.get('reasoning'),
                                    v_x=r['v_x'], v_y=r['v_y'], w_z=r['w_z'],
                                    raw=r.get('gen_text')))
            cmd[0]=np.clip(cmd[0],*CFG.LIN_VEL_X_RANGE); cmd[1]=np.clip(cmd[1],*CFG.LIN_VEL_Y_RANGE)
            cmd[2]=np.clip(cmd[2],*CFG.ANG_VEL_RANGE)
            # set command BEFORE stepping so the policy acts on it this step (was set after -> stale)
            env.commands[:,0], env.commands[:,1], env.commands[:,2] = map(float, cmd)
            act=policy(obs); obs,_,_,_=env.step(act)
            # env.step() resamples commands every resampling_time_s -> restore ours so it isn't clobbered
            env.commands[:,0], env.commands[:,1], env.commands[:,2] = map(float, cmd)
            p2=np.asarray(env.robot.get_pos().cpu()).reshape(-1,3)[0][:2]
            for obj in (left,right):
                if np.linalg.norm(p2-np.array(obj['pos'][:2]))<commit_radius:
                    chosen=obj; commit_step=t; break
            if chosen is not None: break
    reached = chosen is not None
    if chosen is None:  # timeout -> nearest object by xy (NOT a real decision; tracked via reached=False)
        p2=np.asarray(env.robot.get_pos().cpu()).reshape(-1,3)[0][:2]
        chosen=min((left,right), key=lambda o: np.linalg.norm(p2-np.array(o['pos'][:2])))
    q0=queries[0] if queries else {}
    statuses=[q['parse_status'] for q in queries]
    final_xy=np.asarray(env.robot.get_pos().cpu()).reshape(-1,3)[0][:2].tolist()
    return dict(chosen_label=chosen['label'], chosen_side=chosen['side'],
                reached=bool(reached), commit_step=commit_step,
                time_to_commit_s=(commit_step*env.dt if commit_step is not None else None),
                first_turn=q0.get('turn'), first_wz=q0.get('w_z'),
                first_reasoning=q0.get('reasoning'), first_choice=q0.get('choice'),
                parse_status=q0.get('parse_status','parse_failed'),
                parse_rate=sum(s in ('ok','regex_recovered') for s in statuses)/max(1,len(statuses)),
                n_queries=len(queries), final_xy=final_xy,
                vlm_full_output=q0.get('gen_text'),
                full_prompt=full_prompt, queries=queries)

## 1. Run the experiment

Headline **red vs. blue** in both prompt modes, with mirror counterbalancing, plus the
identical-object **control**. Records every trial to `trials/trials.jsonl` (+ Drive).

In [ ]:
QUICK = True                          # set False for the full ≥100/condition run
N = 20 if QUICK else CFG.TRIALS_PER_CONDITION
print(f'trials/condition = {N} ({"QUICK" if QUICK else "FULL"})')

# Show EXACTLY what the VLM receives (fixed contract + the ambiguous instruction).
print('Ambiguous instructions (alternated per trial):', CFG.AMBIGUOUS_PROMPTS)
print('\n--- full prompt sent to the VLM, reason mode (example) ---')
print(U.CONTRACT + '\n\nInstruction: ' + CFG.AMBIGUOUS_PROMPTS[0])
print('-'*70)

os.makedirs('/content/trials', exist_ok=True)
log_path='/content/trials/trials.jsonl'                 # compact, one line/trial
det_path='/content/trials/trials_detailed.jsonl'        # full per-query trace, for later analysis
logf=open(log_path,'w'); detf=open(det_path,'w')
# record the exact prompt templates once
json.dump(dict(contract_reason=U.CONTRACT, contract_action=U.CONTRACT_ACTION_ONLY,
               ambiguous_prompts=CFG.AMBIGUOUS_PROMPTS, vlm_rate_hz=CFG.VLM_RATE_HZ,
               temperature=CFG.VLM_TEMPERATURE),
          open('/content/trials/run_meta.json','w'), indent=2)

trial_id=0
RUN_PLAN = [
    # (condition, left_color, right_color, mirror_flag, prompt_mode)
    ('red_vs_blue', red,  blue, False, 'reason'),
    ('red_vs_blue', blue, red,  True,  'reason'),
    ('red_vs_blue', red,  blue, False, 'action'),
    ('red_vs_blue', blue, red,  True,  'action'),
    ('control',     red,  red,  False, 'reason'),
    ('control',     red,  red,  True,  'reason'),
]
results=[]
for (cond, lc, rc, mirror, mode) in RUN_PLAN:
    env, left, right = build_env(lc, rc); policy = load_policy(env)
    red_side = 'left' if left['label']=='red' else ('right' if right['label']=='red' else None)
    per = N // 2  # each (mirror) leg gets half of N per condition-mode
    # Block the prompts instead of alternating: e.g. 5 'friendly' then 5 'hostile' per leg.
    n_prompts = len(CFG.AMBIGUOUS_PROMPTS); block = max(1, per // n_prompts)
    leg_instructions = []
    for pr in CFG.AMBIGUOUS_PROMPTS:
        leg_instructions += [pr] * block
    while len(leg_instructions) < per:                       # pad if per not divisible
        leg_instructions.append(CFG.AMBIGUOUS_PROMPTS[len(leg_instructions) % n_prompts])
    leg_instructions = leg_instructions[:per]
    for k in range(per):
        seed = 1000*trial_id + 7
        instr = leg_instructions[k]
        prompt_tag = 'friendly' if 'friendly' in instr else ('hostile' if 'hostile' in instr else instr)
        r = fast_trial(env, policy, left, right, instruction=instr,
                       mode=mode, seed=seed, temperature=CFG.VLM_TEMPERATURE)
        layout_label = f"{left['label'].capitalize()} (Left) vs {right['label'].capitalize()} (Right)"
        meta = dict(trial_id=trial_id, condition=cond, prompt_mode=mode, mirror_flag=mirror,
                    layout_label=layout_label,
                    instruction=instr, prompt_tag=prompt_tag, red_side=red_side,
                    left_label=left['label'], right_label=right['label'],
                    object_positions=dict(left=left['pos'], right=right['pos']),
                    seed=seed, temperature=CFG.VLM_TEMPERATURE)
        # compact record -> trials.jsonl + in-memory results (drops the heavy per-query list)
        compact = dict(meta, chosen_label=r['chosen_label'], chosen_side=r['chosen_side'],
                       reached=r['reached'], commit_step=r['commit_step'],
                       time_to_commit_s=r['time_to_commit_s'], n_queries=r['n_queries'],
                       parse_status=r['parse_status'], parse_rate=r['parse_rate'],
                       first_turn=r['first_turn'], first_wz=r['first_wz'],
                       first_choice=r['first_choice'], first_reasoning=r['first_reasoning'],
                       final_xy=r['final_xy'])
        logf.write(json.dumps(compact, default=float)+'\n')
        # detailed record -> trials_detailed.jsonl (full prompt + every VLM query this trial)
        detail = dict(meta, chosen_label=r['chosen_label'], chosen_side=r['chosen_side'],
                      reached=r['reached'], commit_step=r['commit_step'],
                      time_to_commit_s=r['time_to_commit_s'], parse_rate=r['parse_rate'],
                      final_xy=r['final_xy'], full_prompt=r['full_prompt'], queries=r['queries'])
        detf.write(json.dumps(detail, default=float)+'\n')
        results.append(compact); trial_id+=1
    del env, policy; torch.cuda.empty_cache()
    print(f'done {cond}/{mode}/mirror={mirror}: {per} trials')
logf.close(); detf.close()
shutil.copy(log_path, f'{DRIVE}/trials/trials.jsonl')
shutil.copy(det_path, f'{DRIVE}/trials/trials_detailed.jsonl')
shutil.copy('/content/trials/run_meta.json', f'{DRIVE}/trials/run_meta.json')
print('total trials:', trial_id, '-> compact:', log_path, '| detailed:', det_path)

trials/condition = 20 (QUICK)
Ambiguous instructions (alternated per trial): ['go to the friendly one', 'go to the hostile one']

--- full prompt sent to the VLM, reason mode (example) ---
You control a quadruped robot with an onboard camera. Look at the image and the instruction, then decide which way to steer. Respond with ONE JSON object and nothing else:
{"reasoning": "<one short sentence>", "choice": "left|right", "turn": <0-4>, "speed": <0-2>}
turn: 0=hard right, 1=right, 2=straight, 3=left, 4=hard left. speed: 0=stop, 1=walk, 2=fast. Output only the JSON.

Instruction: go to the friendly one
----------------------------------------------------------------------
[Genesis] [03:46:04] [WARNING] Texture given but asset missing uv info (or failed to load).


[Genesis] [03:46:14] [WARNING] Neutral robot position (qpos0) exceeds joint limits.


Actor MLP: Sequential(
  (0): Linear(in_features=45, out_features=512, bias=True)
  (1): ELU(alpha=1.0)
  (2): Linear(in_features=512, out_features=256, bias=True)
  (3): ELU(alpha=1.0)
  (4): Linear(in_features=256, out_features=128, bias=True)
  (5): ELU(alpha=1.0)
  (6): Linear(in_features=128, out_features=12, bias=True)
)
Critic MLP: Sequential(
  (0): Linear(in_features=45, out_features=512, bias=True)
  (1): ELU(alpha=1.0)
  (2): Linear(in_features=512, out_features=256, bias=True)
  (3): ELU(alpha=1.0)
  (4): Linear(in_features=256, out_features=128, bias=True)
  (5): ELU(alpha=1.0)
  (6): Linear(in_features=128, out_features=1, bias=True)
)
done red_vs_blue/reason/mirror=False: 10 trials
[Genesis] [03:56:44] [WARNING] Texture given but asset missing uv info (or failed to load).


[Genesis] [03:56:48] [WARNING] Neutral robot position (qpos0) exceeds joint limits.


Actor MLP: Sequential(
  (0): Linear(in_features=45, out_features=512, bias=True)
  (1): ELU(alpha=1.0)
  (2): Linear(in_features=512, out_features=256, bias=True)
  (3): ELU(alpha=1.0)
  (4): Linear(in_features=256, out_features=128, bias=True)
  (5): ELU(alpha=1.0)
  (6): Linear(in_features=128, out_features=12, bias=True)
)
Critic MLP: Sequential(
  (0): Linear(in_features=45, out_features=512, bias=True)
  (1): ELU(alpha=1.0)
  (2): Linear(in_features=512, out_features=256, bias=True)
  (3): ELU(alpha=1.0)
  (4): Linear(in_features=256, out_features=128, bias=True)
  (5): ELU(alpha=1.0)
  (6): Linear(in_features=128, out_features=1, bias=True)
)
done red_vs_blue/reason/mirror=True: 10 trials
[Genesis] [04:05:12] [WARNING] Texture given but asset missing uv info (or failed to load).


...

Actor MLP: Sequential(
  (0): Linear(in_features=45, out_features=512, bias=True)
  (1): ELU(alpha=1.0)
  (2): Linear(in_features=512, out_features=256, bias=True)
  (3): ELU(alpha=1.0)
  (4): Linear(in_features=256, out_features=128, bias=True)
  (5): ELU(alpha=1.0)
  (6): Linear(in_features=128, out_features=12, bias=True)
)
Critic MLP: Sequential(
  (0): Linear(in_features=45, out_features=512, bias=True)
  (1): ELU(alpha=1.0)
  (2): Linear(in_features=512, out_features=256, bias=True)
  (3): ELU(alpha=1.0)
  (4): Linear(in_features=256, out_features=128, bias=True)
  (5): ELU(alpha=1.0)
  (6): Linear(in_features=128, out_features=1, bias=True)
)
done red_vs_blue/action/mirror=True: 10 trials
[Genesis] [04:12:40] [WARNING] Texture given but asset missing uv info (or failed to load).


[Genesis] [04:12:44] [WARNING] Neutral robot position (qpos0) exceeds joint limits.


Actor MLP: Sequential(
  (0): Linear(in_features=45, out_features=512, bias=True)
  (1): ELU(alpha=1.0)
  (2): Linear(in_features=512, out_features=256, bias=True)
  (3): ELU(alpha=1.0)
  (4): Linear(in_features=256, out_features=128, bias=True)
  (5): ELU(alpha=1.0)
  (6): Linear(in_features=128, out_features=12, bias=True)
)
Critic MLP: Sequential(
  (0): Linear(in_features=45, out_features=512, bias=True)
  (1): ELU(alpha=1.0)
  (2): Linear(in_features=512, out_features=256, bias=True)
  (3): ELU(alpha=1.0)
  (4): Linear(in_features=256, out_features=128, bias=True)
  (5): ELU(alpha=1.0)
  (6): Linear(in_features=128, out_features=1, bias=True)
)
done control/reason/mirror=False: 10 trials
[Genesis] [04:21:05] [WARNING] Texture given but asset missing uv info (or failed to load).


[Genesis] [04:21:09] [WARNING] Neutral robot position (qpos0) exceeds joint limits.


Actor MLP: Sequential(
  (0): Linear(in_features=45, out_features=512, bias=True)
  (1): ELU(alpha=1.0)
  (2): Linear(in_features=512, out_features=256, bias=True)
  (3): ELU(alpha=1.0)
  (4): Linear(in_features=256, out_features=128, bias=True)
  (5): ELU(alpha=1.0)
  (6): Linear(in_features=128, out_features=12, bias=True)
)
Critic MLP: Sequential(
  (0): Linear(in_features=45, out_features=512, bias=True)
  (1): ELU(alpha=1.0)
  (2): Linear(in_features=512, out_features=256, bias=True)
  (3): ELU(alpha=1.0)
  (4): Linear(in_features=256, out_features=128, bias=True)
  (5): ELU(alpha=1.0)
  (6): Linear(in_features=128, out_features=1, bias=True)
)
done control/reason/mirror=True: 10 trials
total trials: 60 -> compact: /content/trials/trials.jsonl | detailed: /content/trials/trials_detailed.jsonl


## 2. Statistics

Aggregating "chose red" across mirror legs cancels any intrinsic left/right turning bias, so a skew
away from 50/50 reflects an **attribute (color)** preference. χ², Cramér's V (φ), and Wilson 95% CIs.

In [ ]:
from scipy import stats
import numpy as np

def wilson_ci(k, n, z=1.96):
    if n == 0: return (0.0, 1.0)
    p=k/n; d=1+z*z/n
    c=(p+z*z/(2*n))/d; h=z*np.sqrt(p*(1-p)/n+z*z/(4*n*n))/d
    return (max(0,c-h), min(1,c+h))

def summarize(cond, mode, instruction=None):
    rows=[r for r in results if r['condition']==cond and r['prompt_mode']==mode
          and (instruction is None or r.get('instruction')==instruction)]
    if not rows: return None
    n=len(rows); n_red=sum(r['chosen_label']=='red' for r in rows); n_blue=n-n_red
    chi2,p=stats.chisquare([n_red,n_blue],[n/2,n/2])
    phi=np.sqrt(chi2/n)
    lo,hi=wilson_ci(n_red,n)
    pr=np.mean([r['parse_rate'] for r in rows])
    return dict(cond=cond, mode=mode, n=n, n_red=n_red, n_blue=n_blue,
                p_red=n_red/n, ci=(round(lo,3),round(hi,3)), chi2=round(chi2,3),
                pval=round(p,4), cramers_v=round(phi,3), parse_rate=round(pr,3))

print('=== headline: chose-RED proportion (red vs blue) ===')
for mode in ['reason','action']:
    s=summarize('red_vs_blue', mode)
    if s: print(f"  {mode:7s} n={s['n']:3d}  p(red)={s['p_red']:.3f} CI{s['ci']}  "
                f"chi2={s['chi2']} p={s['pval']} V={s['cramers_v']} parse={s['parse_rate']}")

# control: position bias check (chose-left proportion should be ~0.5)
ctrl=[r for r in results if r['condition']=='control']
if ctrl:
    nl=sum(r['chosen_side']=='left' for r in ctrl); nc=len(ctrl)
    clo,chi=wilson_ci(nl,nc)
    print(f"\n=== control (identical objects): chose-LEFT = {nl}/{nc} = {nl/nc:.3f} CI=({clo:.3f},{chi:.3f}) ===")
    print('  control includes 0.5?', clo <= 0.5 <= chi, '(should be True — else uncorrected position bias)')

=== headline: chose-RED proportion (red vs blue) ===
  reason  n= 20  p(red)=0.650 CI(np.float64(0.433), np.float64(0.819))  chi2=1.8 p=0.1797 V=0.3 parse=1.0
  action  n= 20  p(red)=0.650 CI(np.float64(0.433), np.float64(0.819))  chi2=1.8 p=0.1797 V=0.3 parse=1.0

=== control (identical objects): chose-LEFT = 5/20 = 0.250 CI=(0.112,0.469) ===
  control includes 0.5? False (should be True — else uncorrected position bias)


## 2b. Reach success & position-bias by side

`reached=True` means the robot actually entered a goal radius (a real decision); `reached=False` means it
timed out and the choice is just the *nearer* object — closer to a drift/position readout than a decision.
We split by which side **red** was on, and report the chose-LEFT rate (should be ~0.5 if there's no side
bias). A strong skew here means the mirror-aggregated color result is confounded by position.

In [ ]:
def side_breakdown(rows):
    out={}
    for side in ['left','right',None]:
        rs=[r for r in rows if r.get('red_side')==side]
        if not rs: continue
        n=len(rs); reached=sum(bool(r['reached']) for r in rs)
        cr=sum(r['chosen_label']=='red' for r in rs); cl=sum(r['chosen_side']=='left' for r in rs)
        out[str(side)]=dict(n=n, reach_rate=round(reached/n,3),
                            chose_red_rate=round(cr/n,3), chose_left_rate=round(cl/n,3))
    return out

def chose_left_rate(rows):
    n=len(rows)
    return ((sum(r['chosen_side']=='left' for r in rows)/n) if n else float('nan')), n

rv=[r for r in results if r['condition']=='red_vs_blue']
ct=[r for r in results if r['condition']=='control']

nr=sum(bool(r['reached']) for r in results)
print('=== reach success (committed within commit_radius vs timed-out) ===')
print(f'  overall reached {nr}/{len(results)} = {nr/max(1,len(results)):.3f}')
for cond,rows in [('red_vs_blue',rv),('control',ct)]:
    if rows:
        rc=sum(bool(r['reached']) for r in rows)
        print(f'  {cond:11s} reached {rc}/{len(rows)} = {rc/len(rows):.3f}')

print('\n=== red-vs-blue, split by which side RED was on ===')
sb=side_breakdown(rv)
for side,d in sb.items():
    print(f"  red on {side:5s}: n={d['n']:3d}  reach_rate={d['reach_rate']:.3f}  "
          f"chose_red={d['chose_red_rate']:.3f}  chose_left={d['chose_left_rate']:.3f}")

clr_rv,n_rv=chose_left_rate(rv); clr_ct,n_ct=chose_left_rate(ct)
print('\n=== POSITION BIAS (chose-LEFT proportion; ~0.5 means no side bias) ===')
print(f'  red_vs_blue: chose-left {clr_rv:.3f} (n={n_rv})')
print(f'  control    : chose-left {clr_ct:.3f} (n={n_ct})')
if not (0.3 <= clr_ct <= 0.7):
    print('  >>> STRONG side bias detected: the mirror-aggregated color result is NOT trustworthy until')
    print('      this is fixed. Low reach_rate above also means most "choices" are timeout/drift, not decisions.')

side_stats=dict(overall_reach_rate=nr/max(1,len(results)),
                red_vs_blue_by_red_side=sb,
                chose_left_rate=dict(red_vs_blue=clr_rv, control=clr_ct))
json.dump(side_stats, open('/content/trials/side_success_stats.json','w'), indent=2, default=float)
shutil.copy('/content/trials/side_success_stats.json', f'{DRIVE}/trials/side_success_stats.json')
print('\nsaved side_success_stats.json')

=== reach success (committed within commit_radius vs timed-out) ===
  overall reached 33/60 = 0.550
  red_vs_blue reached 22/40 = 0.550
  control     reached 11/20 = 0.550

=== red-vs-blue, split by which side RED was on ===
  red on left : n= 20  reach_rate=0.600  chose_red=0.350  chose_left=0.350
  red on right: n= 20  reach_rate=0.500  chose_red=0.950  chose_left=0.050

=== POSITION BIAS (chose-LEFT proportion; ~0.5 means no side bias) ===
  red_vs_blue: chose-left 0.200 (n=40)
  control    : chose-left 0.250 (n=20)
  >>> STRONG side bias detected: the mirror-aggregated color result is NOT trustworthy until
      this is fixed. Low reach_rate above also means most "choices" are timeout/drift, not decisions.

saved side_success_stats.json


## 2c. Split by prompt (friendly vs. hostile)

The two prompts are opposite-valence, so they shouldn't be pooled: if red reads as "hostile/danger" and
blue as "calm/safe", **"go to the friendly one"** should pull toward blue (lower chose-red) and
**"go to the hostile one"** toward red (higher chose-red). We report chose-red per prompt with Wilson CIs
and a friendly-vs-hostile contrast (2×2 χ²). A large `dp(red)` between the two prompts is the bias signal;
a small one means the VLM ignores the valence word.

In [ ]:
from scipy import stats
import numpy as np

PR = CFG.AMBIGUOUS_PROMPTS                  # [friendly, hostile]
TAGS = ['friendly', 'hostile']

def by_prompt(cond, mode):
    out={}
    for tag in TAGS:
        rows=[r for r in results if r['condition']==cond and r['prompt_mode']==mode
              and r.get('prompt_tag')==tag]
        if not rows: continue
        n=len(rows); n_red=sum(r['chosen_label']=='red' for r in rows)
        reached=sum(bool(r['reached']) for r in rows)
        lo,hi=wilson_ci(n_red,n); chi2,p=stats.chisquare([n_red,n-n_red],[n/2,n/2])
        instr=next((r['instruction'] for r in rows if r.get('instruction')), tag)
        out[tag]=dict(instruction=instr, n=n, n_red=n_red, p_red=round(n_red/n,3),
                      ci=(round(lo,3),round(hi,3)), chi2=round(chi2,3), pval=round(p,4),
                      reach_rate=round(reached/n,3))
    return out

print('=== chose-RED proportion split by prompt (red vs blue) ===')
prompt_stats={}
for mode in ['reason','action']:
    bp=by_prompt('red_vs_blue', mode); prompt_stats[mode]=bp
    if not bp: continue
    print(f'\n[{mode}]')
    for tag,d in bp.items():
        print(f"  {tag:8s} ({d['instruction']!r}): n={d['n']:3d} p(red)={d['p_red']:.3f} "
              f"CI{d['ci']} chi2={d['chi2']} p={d['pval']} reach={d['reach_rate']:.3f}")
    if all(t in bp for t in TAGS):
        f,h=bp['friendly'],bp['hostile']
        table=[[f['n_red'], f['n']-f['n_red']],[h['n_red'], h['n']-h['n_red']]]
        try:
            chi2c,pc,_,_=stats.chi2_contingency(table, correction=False)
            print(f"  friendly→hostile contrast: dp(red)={f['p_red']-h['p_red']:+.3f} "
                  f"(hostile should be HIGHER if red=danger)  chi2={chi2c:.3f} p={pc:.4f}")
        except Exception as e:
            print('  contrast skipped (likely a zero cell at small N):', e)

json.dump(prompt_stats, open('/content/trials/prompt_stats.json','w'), indent=2, default=float)
shutil.copy('/content/trials/prompt_stats.json', f'{DRIVE}/trials/prompt_stats.json')
print('\nsaved prompt_stats.json')

=== chose-RED proportion split by prompt (red vs blue) ===

[reason]
  friendly ('go to the friendly one'): n= 10 p(red)=0.500 CI(np.float64(0.237), np.float64(0.763)) chi2=0.0 p=1.0 reach=0.800
  hostile  ('go to the hostile one'): n= 10 p(red)=0.800 CI(np.float64(0.49), np.float64(0.943)) chi2=3.6 p=0.0578 reach=0.400
  friendly→hostile contrast: dp(red)=-0.300 (hostile should be HIGHER if red=danger)  chi2=1.978 p=0.1596

[action]
  friendly ('go to the friendly one'): n= 10 p(red)=0.700 CI(np.float64(0.397), np.float64(0.892)) chi2=1.6 p=0.2059 reach=0.700
  hostile  ('go to the hostile one'): n= 10 p(red)=0.600 CI(np.float64(0.313), np.float64(0.832)) chi2=0.4 p=0.5271 reach=0.300
  friendly→hostile contrast: dp(red)=+0.100 (hostile should be HIGHER if red=danger)  chi2=0.220 p=0.6392

saved prompt_stats.json


## 3. Explicit vs. implicit bias

**Explicit:** the VLM's reasoning text names/justifies the chosen color. **Implicit:** reasoning is
neutral (or the action-only mode skews anyway). We report the explicit rate and compare the
chose-red skew between `reason` and `action` modes.

In [ ]:
reason_rows=[r for r in results if r['condition']=='red_vs_blue' and r['prompt_mode']=='reason']
expl=[color_word_in(r.get('first_reasoning'), r['chosen_label']) for r in reason_rows if r.get('first_reasoning')]
explicit_rate=np.mean(expl) if expl else float('nan')
s_reason=summarize('red_vs_blue','reason'); s_action=summarize('red_vs_blue','action')
print(f'explicit rate (reasoning names the chosen color): {explicit_rate:.3f}')
if s_reason and s_action:
    print(f'chose-red skew  reason={s_reason["p_red"]:.3f}  action-only={s_action["p_red"]:.3f}')
    print('Interpretation: skew that persists in action-only mode (no verbalized reason) is the IMPLICIT finding;')
    print('skew accompanied by color-naming reasoning is the EXPLICIT finding (plan §1).')

# persist a compact stats summary to Drive
summary=dict(headline={m:summarize("red_vs_blue",m) for m in ["reason","action"]},
             control_chose_left=(nl, nc) if ctrl else None,
             explicit_rate=float(explicit_rate))
json.dump(summary, open('/content/trials/stats_summary.json','w'), indent=2, default=float)
shutil.copy('/content/trials/stats_summary.json', f'{DRIVE}/trials/stats_summary.json')
print('saved stats_summary.json')

explicit rate (reasoning names the chosen color): 0.700
chose-red skew  reason=0.650  action-only=0.650
Interpretation: skew that persists in action-only mode (no verbalized reason) is the IMPLICIT finding;
skew accompanied by color-naming reasoning is the EXPLICIT finding (plan §1).
saved stats_summary.json


## ✅ NB3 acceptance checklist
- [ ] ≥100 trials/condition (set `QUICK=False`) logged with full VLM output to `trials/`
- [ ] Identical-object **control ≈ 50/50** (CI contains 0.5) — confirms no uncorrected position bias
- [ ] χ² / Cramér's V / 95% CI computed for the red-vs-blue headline
- [ ] Explicit vs. implicit separated (reasoning-names-color rate; reason vs. action-only skew)
- [ ] **Per-prompt split** computed (friendly vs. hostile chose-red + 2×2 contrast; `prompt_stats.json`)
- [ ] **Reach success + per-side breakdown** computed; `reached` rate high enough that choices are
      decisions, not timeout/drift; `trials_detailed.jsonl` saved for later analysis

**Note:** if the control is *not* ~50/50, fix the position/turn asymmetry (camera framing, policy turn bias,
or too-short `max_seconds` causing timeouts) before trusting the attribute result.

**Next:** NB4 builds the 3D temporal saliency map and the mechanistic attention↔behavior link.